## If working in Google Colab, run this first
Run the cell below at the start of every session. It downloads the course files to your Google Drive and keeps them up to date.
When you see "Setup complete", you are ready to run the rest of the notebook.

**Do not edit or save your work inside the `AIforMovementScience` folder.**
Use **File > Save a copy in Drive** and work on your own copy.

In [ ]:
# [course-setup] Run this cell first, every session (or use Runtime > Run all)
import os, sys
try:
    from google.colab import drive
    os.chdir('/content')                  # start from a valid directory
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/AIforMovementScience'
    if os.path.isdir(REPO + '/.git'):
        # already downloaded before: just update it
        !git -C "$REPO" fetch origin main && git -C "$REPO" reset --hard origin/main
    else:
        # first time, or a broken/partial copy: download a fresh copy
        !rm -rf "$REPO"
        !git clone https://github.com/dguari1/AIforMovementScience.git "$REPO"
except ImportError:
    # not in Colab: walk up from here until the repository root is found
    REPO = os.getcwd()
    while REPO != '/' and not os.path.isdir(os.path.join(REPO, 'lectures', 'data')):
        REPO = os.path.dirname(REPO)

WORKDIR = REPO + '/lectures'
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print('Setup complete. You can now run the rest of the notebook.')

# Classification of Clinical Gait Data: GMFCS Level in Cerebral Palsy

In this notebook, we compare the classifiers from notebook 04.02 on a real clinical gait dataset.

The data comes from this paper:

>Kidziński, Ł., Yang, B., Hicks, J. L., Rajagopal, A., Delp, S. L., & Schwartz, M. H. (2020). Deep neural networks enable quantitative movement analysis using single-camera videos. *Nature Communications*, 11(1), 4054.

The patients were children and young adults with cerebral palsy (CP). A motion capture system recorded their gait, and the gait laboratory computed many gait parameters. Some parameters have a left and a right value (for example the knee flexion), and other parameters do not depend on the side (for example the speed and the cadence). Many patients came to the laboratory for **more than one visit**, so one patient can have many rows.

A clinician also gave each patient a **Gross Motor Function Classification System (GMFCS)** level. GMFCS is a 5-level scale of the mobility of children and youth with CP:

| Level | Mobility |
| --- | --- |
| I | walks without limitations |
| II | walks with limitations |
| III | walks with a hand-held mobility device (for example a walker or crutches) |
| IV | self-mobility with limitations; may use powered mobility |
| V | transported in a manual wheelchair |

We will make a binary classification problem:
- **Class 0, GMFCS I–II**: walks without a mobility device
- **Class 1, GMFCS III–IV**: walks with a mobility device

## Learning goals
1. Make a clinically meaningful binary target from the GMFCS level, and check the data first.
2. Use a **subject-aware** train/test split to prevent data leakage.
3. Use **group-aware cross-validation** inside a `Pipeline` to tune the hyperparameters.
4. Compare logistic regression, a decision tree, a random forest and kNN.
5. Evaluate the models with confusion matrices and ROC curves.
6. Interpret the models: coefficients, the tree plot, and the feature importance of the forest.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (roc_auc_score, roc_curve, balanced_accuracy_score,
                             classification_report, ConfusionMatrixDisplay)

In [ ]:
# read the data from the csv file
df = pd.read_csv('data/GMFCS_classification_data.csv', index_col=0)
print("Shape:", df.shape)
df.head()

In [ ]:
print('Total number of rows:', len(df))
print('Unique patients:', df["Patient_ID"].nunique())
subject_counts = df["Patient_ID"].value_counts()
print(f"Rows per patient: median {subject_counts.median():.0f}, min {subject_counts.min()}, max {subject_counts.max()}")

print("\nRows for each value of gmfcs:")
print(df["gmfcs"].value_counts(dropna=False).sort_index())

## Check the labels before you use them

GMFCS has the levels 1 to 5. But the `gmfcs` column also has the codes **0** and **6**, and it has no level 5 (these children do not walk, so they have no gait data).

What do the codes 0 and 6 mean? The file does not tell us. A good first check is to compare a simple gait measure, the walking speed, between the codes:

In [ ]:
df.groupby("gmfcs")["speed"].agg(['count', 'mean']).round(2)

The speed decreases from level 1 to level 4, as we expect. But the rows with code 0 walk **slower** than GMFCS level 1. So code 0 is not a group of "unimpaired" children. It is probably a code for "not rated". Code 6 is not a GMFCS level.

**We do not know what these codes mean, so we remove them.** Always check that a label means what you think it means before you train a model on it.

In [ ]:
# keep only the real GMFCS levels 1 to 4
df_model = df[df["gmfcs"].isin([1, 2, 3, 4])].copy()

# class 0 = GMFCS I-II (walks without a device), class 1 = GMFCS III-IV (walks with a device)
df_model["target"] = (df_model["gmfcs"] >= 3).astype(int)
print(df_model["target"].value_counts().sort_index())

The classes are not balanced: about one third of the rows are in class 1. Remember this when you read the accuracy.

## Select the features and check the values

In [ ]:
features = [
    'age', 'height', 'mass', 'bmi',
    'cadence', 'speed', 'steplen',
    'HipFlex_IC_L', 'HipFlex_IC_R',
    'HipRot_mean_L', 'HipRot_mean_R',
    'KneeFlex_maxExtension_L', 'KneeFlex_maxExtension_R',
    'KneeFlex_meanStance_L', 'KneeFlex_meanStance_R'
]

# a new DataFrame with the subject id (for grouping), the target and the features
model_df = df_model[["Patient_ID", "target"] + features].dropna()
model_df[['age', 'height', 'mass', 'bmi', 'speed']].describe().round(1)

Look at the minimum and maximum values. Some values are not possible for a person (for example a very small height or a very large BMI). These are data entry errors. We remove the rows with impossible values.

In [ ]:
n_before = len(model_df)
model_df = model_df[model_df['height'].between(80, 200) & model_df['bmi'].between(10, 45)]
print(f"Removed {n_before - len(model_df)} rows. Rows left: {len(model_df)}, patients: {model_df['Patient_ID'].nunique()}")

## Why do we need group-aware data partitioning?

In many biomechanics and clinical datasets, we have **multiple observations from the same subject**.  
For example, a dataset may include:

- left and right limb measurements  
- repeated visits  
- multiple trials  
- different tasks performed by the same participant  

Because these observations come from the **same individual**, they are not fully independent.

---

### The problem with random row-wise splitting

If we randomly split the dataset by rows, it is very likely that:

- some observations from a subject will be placed in the **training set**
- other observations from the same subject will be placed in the **testing set**

This creates a problem called **data leakage**.

The model may learn **subject-specific patterns** (for example, walking style, body size, or movement variability) instead of learning general patterns that distinguish the classes.

As a result:

- the model may appear to perform very well on the test set  
- but this performance is overly optimistic  
- the model may perform worse when applied to **new subjects**

In other words, we do not test generalization. We partly test memory.

---

### What is group-aware partitioning?

Group-aware partitioning ensures that **all observations from the same subject are kept together**.

For example:

- all rows from Subject A → training set  
- all rows from Subject B → testing set  

This means that the model is evaluated on **completely unseen individuals**, which provides a more realistic estimate of performance.

Common tools include:

- `GroupShuffleSplit` for train/test splitting  
- `GroupKFold` for cross-validation  

In both cases, we specify a grouping variable such as `Patient_ID`.

---

### Why is this important in movement science?

In biomechanics and clinical movement analysis:

- individuals have characteristic movement patterns  
- repeated measures are common  
- datasets are often hierarchical (trials within sessions within subjects)

If we ignore this structure:

- model performance can be artificially inflated  
- conclusions about clinical usefulness may be misleading  

Using group-aware partitioning helps ensure that:

> the model is learning patterns that generalize across people, not just patterns that identify specific individuals.

---


When multiple observations come from the same subject,  
**split the data by subject, not by row.**

This leads to more honest model evaluation and better understanding of how the model will perform in real-world applications.

In [ ]:
# the feature table, the target vector and the subject ids (the groups)
X = model_df[features]
y = model_df["target"].values
groups = model_df["Patient_ID"].values

# GroupShuffleSplit keeps all the rows of one patient in the same set
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
groups_train = groups[train_idx]

print("Training rows:", len(X_train), "| Testing rows:", len(X_test))
print("Training patients:", len(set(groups_train)), "| Testing patients:", len(set(groups[test_idx])))
print("Patients in both sets:", len(set(groups_train) & set(groups[test_idx])))

## A first model: logistic regression

The features have very different units (years, cm, kg, degrees), so we scale them. We put the scaler and the model in a `Pipeline`. The pipeline fits the scaler on the training data only, and then applies the same scaling to the test data. This is important for cross-validation later: in each fold, the scaler sees only the training part of that fold.

In [ ]:
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=5000))
])
lr_pipe.fit(X_train, y_train)
y_pred = lr_pipe.predict(X_test)

print(classification_report(y_test, y_pred, target_names=['GMFCS I-II', 'GMFCS III-IV']))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['I-II', 'III-IV'], cmap='Blues')
plt.title('Logistic regression (test set)')
plt.show()

Because the features are scaled, all the features have a mean of 0 and a standard deviation of 1. So we can compare the sizes of the coefficients. A positive coefficient increases the probability of GMFCS III–IV.

In [ ]:
coef_df = pd.DataFrame({
    "feature": features,
    "coefficient": lr_pipe.named_steps['model'].coef_[0]
}).sort_values("coefficient")

plt.figure(figsize=(8, 6))
colors = ["tab:red" if v < 0 else "tab:blue" for v in coef_df["coefficient"]]
plt.barh(coef_df["feature"], coef_df["coefficient"], color=colors)
plt.axvline(0, color="black", linewidth=1)
plt.title("Logistic regression coefficients (scaled features)")
plt.xlabel("Coefficient value")
plt.tight_layout()
plt.show()

The speed has the largest negative coefficient: a slower patient is more probably in GMFCS III–IV. This agrees with the table of speeds above.

**Be careful:** `steplen` has a *positive* coefficient, but a short step is a sign of impairment. The reason is that speed, step length and cadence are strongly correlated (speed = step length × cadence). When inputs are correlated, the model can divide their effect in strange ways (Assignment 6). Read the coefficients of correlated inputs with care.

## Why do we need hyperparameter tuning?

Most machine-learning models have settings called **hyperparameters** that control how the model learns. Examples:

- the depth of a decision tree
- the number of trees in a random forest
- the number of neighbors in kNN
- the regularization strength `C` in logistic regression

The model does **not** learn these settings from the data. We must select them.

### Why does this matter?

Different hyperparameter values can give very different models:

- a very deep decision tree can **overfit** the training data
- a very shallow tree can **underfit** and miss important patterns
- too few neighbors in kNN make the predictions noisy
- too many neighbors make the model too rigid

We want the values that give a model that works well on **new patients**.

### How do we tune hyperparameters?

1. Make a list of values to try for each hyperparameter (a **grid**).
2. For each combination, run **cross-validation on the training data** and compute a metric (here, the ROC AUC).
3. Select the combination with the best mean metric.

`GridSearchCV` does these steps for us. We use `GroupKFold`, so the patients in each validation fold are not in the training part of that fold. **The test set is not used in this step.** We use the test set only one time, at the end, to evaluate the selected model.

In [ ]:
cv = GroupKFold(n_splits=5)
results = {}   # we keep the tuned models here

def tune_and_test(name, estimator, param_grid):
    # tune with group-aware cross-validation on the training data, then evaluate one time on the test set
    search = GridSearchCV(estimator, param_grid, cv=cv, scoring="roc_auc", n_jobs=-1)
    search.fit(X_train, y_train, groups=groups_train)
    best = search.best_estimator_
    prob = best.predict_proba(X_test)[:, 1]
    results[name] = {'model': best, 'prob': prob, 'pred': best.predict(X_test)}
    print(f"{name}")
    print(f"  best hyperparameters: {search.best_params_}")
    print(f"  cross-validation AUC (training data): {search.best_score_:.3f}")
    print(f"  test AUC: {roc_auc_score(y_test, prob):.3f}")
    return best

best_lr = tune_and_test("Logistic regression", lr_pipe,
                        {'model__C': [0.01, 0.1, 1, 10, 100], 'model__class_weight': [None, 'balanced']})

### Decision tree

A tree does not use distances or coefficients, so it does not need scaled features.

In [ ]:
best_tree = tune_and_test("Decision tree", DecisionTreeClassifier(random_state=0),
                          {'max_depth': [2, 3, 4, 6, 8, None], 'min_samples_leaf': [1, 5, 20]})

# plot only the first 3 levels of the tree, so that we can read it
plt.figure(figsize=(20, 8))
plot_tree(best_tree, max_depth=2, feature_names=features, class_names=['I-II', 'III-IV'],
          filled=True, rounded=True, fontsize=9)
plt.title("First levels of the tuned decision tree")
plt.show()

The first questions of the tree use speed and cadence. A clinician could follow these rules by hand. This is the main advantage of a single tree: it is easy to read.

### Random forest

In [ ]:
best_rf = tune_and_test("Random forest", RandomForestClassifier(random_state=0, n_jobs=-1),
                        {'n_estimators': [100, 300], 'max_depth': [5, 10, None], 'min_samples_leaf': [1, 5]})

### k-nearest neighbors

kNN uses distances, so the scaler must be in the pipeline.

In [ ]:
knn_pipe = Pipeline([('scaler', StandardScaler()), ('model', KNeighborsClassifier())])
best_knn = tune_and_test("kNN", knn_pipe, {'model__n_neighbors': [1, 5, 15, 31, 51]})

## Compare the models on the test set

In [ ]:
labels = ['I-II', 'III-IV']
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
rows = []
for ax, (name, r) in zip(axes, results.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, r['pred'], display_labels=labels, cmap='Blues', ax=ax, colorbar=False)
    ax.set_title(name)
    rows.append({'Model': name,
                 'ROC AUC': roc_auc_score(y_test, r['prob']),
                 'Balanced accuracy': balanced_accuracy_score(y_test, r['pred'])})
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 5))
for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_test, r['prob'])
    plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {roc_auc_score(y_test, r['prob']):.3f})")
plt.plot([0, 1], [0, 1], 'k--', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC curves (test set)')
plt.legend(loc='lower right', fontsize=8)
plt.grid(alpha=0.3)
plt.show()

pd.DataFrame(rows).set_index('Model').round(3)

**Balanced accuracy** is the mean of the recall of each class. It does not change when one class is more common, so it is better than the accuracy for these unbalanced classes.

## Feature importance of the random forest

In [ ]:
importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": best_rf.feature_importances_
}).sort_values("Importance", ascending=False)

plt.figure(figsize=(8, 5))
plt.barh(importance_df["Feature"], importance_df["Importance"])
plt.gca().invert_yaxis()
plt.xlabel("Feature importance")
plt.title("Random forest feature importance")
plt.grid(True, axis="x", alpha=0.3)
plt.show()

### Can we use fewer features?

We keep the features with an importance above 0.05 and tune the forest again. The forest learned the importance from the **training data only**, so the test set is still unseen.

In [ ]:
selected_features = importance_df.loc[importance_df["Importance"] > 0.05, "Feature"].tolist()
print("Selected features:", selected_features)

search = GridSearchCV(RandomForestClassifier(random_state=0, n_jobs=-1),
                      {'n_estimators': [100, 300], 'max_depth': [5, 10, None], 'min_samples_leaf': [1, 5]},
                      cv=cv, scoring="roc_auc", n_jobs=-1)
search.fit(X_train[selected_features], y_train, groups=groups_train)
prob_sel = search.best_estimator_.predict_proba(X_test[selected_features])[:, 1]
print(f"Random forest, {len(selected_features)} features: test AUC = {roc_auc_score(y_test, prob_sel):.3f}")
print(f"Random forest, {len(features)} features: test AUC = {roc_auc_score(y_test, results['Random forest']['prob']):.3f}")

### How much does speed alone tell us?

Speed is the most important feature for every model. How good is a model that uses **only** the speed?

In [ ]:
speed_only = LogisticRegression().fit(X_train[['speed']], y_train)
prob_speed = speed_only.predict_proba(X_test[['speed']])[:, 1]
print(f"Logistic regression, speed only: test AUC = {roc_auc_score(y_test, prob_speed):.3f}")

## Key points

- **Check the labels first.** The codes 0 and 6 are not GMFCS levels. With the original target (code 0 vs all other codes), the best test AUC was about 0.74. With a clear clinical target (I–II vs III–IV), all the models have a test AUC of about 0.94 to 0.96.
- **All four models are similar.** The random forest has the best test AUC (0.956), but logistic regression (0.952) is almost as good and is easier to explain.
- **AUC and balanced accuracy can disagree.** kNN has a high AUC (0.952) but the lowest balanced accuracy (0.815). Its probabilities rank the patients well, but the default threshold of 0.5 misses many GMFCS III–IV patients. The AUC uses all thresholds; the confusion matrix uses one threshold.
- **Group-aware cross-validation gave an honest estimate.** The cross-validation AUC on the training data (0.92 to 0.94) is close to the test AUC. No patient was in both the training and the validation parts.
- **A simpler model is almost as good.** The forest with 3 features (speed, cadence, step length) has a test AUC of 0.945.
- **Speed alone gives a test AUC of 0.941.** This is expected: patients in GMFCS III–IV use a mobility device and walk slowly. So most of the information in these models is the walking speed. Before you say that a model "detects impairment from the joint angles", check what a speed-only model can do. You will need this check in the final project.

### Try it
Remove `speed`, `cadence` and `steplen` from the list of features and run the notebook again. How much information do the joint angles and the body size give without the speed?